In [ ]:
%pip install openai scikit-learn pandas

In [ ]:
# Create client

from openai import OpenAI
from sklearn.model_selection import train_test_split
import pandas as pd

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",  # Ollama
    api_key="sk-1234"  # Dummy key
)

In [ ]:
# Set model parameters

SYSTEM_PROMPT = """
You are a QIDS-SR (Quick Inventory of Depressive Symptomatology - Self Report) scoring assistant.
Your job is to extract item-level scores for the QIDS-SR based on a given conversation transcript.
The QIDS-SR assesses 16 individual items covering sleep, mood, appetite, weight, concentration, self-view, death/suicide ideation, interest, energy, and psychomotor symptoms over the past seven days.

QIDS-SR item definitions and scale:
item1  (falling asleep):   0=no delay, 1=<30min less than half nights, 2=30-60min more than half nights, 3=>60min more than half nights
item2  (sleep during night): 0=no waking, 1=restless light sleep, 2=wakes 1-2x/night, 3=awake most of night
item3  (waking early):     0=no early waking, 1=<1hr early less than half nights, 2=>=1hr early more than half nights, 3=>=2hr early every night
item4  (sleeping too much): 0=no excess sleep, 1=sleeps 1hr more than usual, 2=sleeps 2hr more than usual, 3=sleeps 3+hr more than usual
item5  (sad mood):         0=no sadness, 1=sad less than half the time, 2=sad more than half the time, 3=sad nearly all the time
item6  (decreased appetite): 0=no change, 1=eats somewhat less, 2=eats much less, 3=rarely eats within 24hr
item7  (increased appetite): 0=no change, 1=eats somewhat more, 2=eats much more, 3=compelled to eat much more
item8  (decreased weight):  0=no change, 1=1-2lb decrease, 2=3-4lb decrease, 3=>5lb decrease
item9  (increased weight):  0=no change, 1=1-2lb increase, 2=3-4lb increase, 3=>5lb increase
item10 (concentration):    0=no change, 1=occasional difficulty, 2=difficulty most of the time, 3=unable to concentrate
item11 (view of self):     0=no change, 1=self-critical more than usual, 2=believes they cause problems, 3=constantly believes they are a failure
item12 (death/suicide):    0=no thoughts, 1=feels life is empty, 2=thinks of death/suicide several times/week, 3=thinks of death/suicide daily
item13 (interest):         0=no change, 1=less interested in some activities, 2=interested in only 1-2 activities, 3=no interest in any activity
item14 (energy):           0=no change, 1=tires more easily, 2=effort needed for routine activities, 3=cannot carry out routine activities
item15 (slowed down):      0=no slowing, 1=slightly slow, 2=clearly slow, 3=cannot move without assistance
item16 (restless):         0=no restlessness, 1=fidgety, 2=wringing hands/can't sit still, 3=moving constantly, can't stay seated

Important scoring rules:
- For sleep domain: only one of items 1-4 will be non-zero (use the dominant sleep disturbance)
- For appetite/weight domain: only items 6-7 OR items 8-9 will be non-zero (not both directions)
- For psychomotor domain: only one of items 15-16 will be non-zero (slowing OR restlessness)

Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The score for each item must be a single string value: "0", "1", "2", or "3".
Rules:
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any item1-item16 key.
Return JSON in exactly this shape:
{
  "item1": "0",
  "item2": "0",
  "item3": "0",
  "item4": "0",
  "item5": "0",
  "item6": "0",
  "item7": "0",
  "item8": "0",
  "item9": "0",
  "item10": "0",
  "item11": "0",
  "item12": "0",
  "item13": "0",
  "item14": "0",
  "item15": "0",
  "item16": "0"
}
"""
MODEL = 'qwen3:8b'

- Use a json dataset of QIDS-SR conversation transcripts
- Extract item-level scores using LLM
- Validate whether they were correct using MSE
- Note: domain scores (D1-D9) can be derived from item scores post-hoc

In [ ]:
# Load the QIDS-SR dataset
import json

with open('qids_valid_dataset.json') as f:
    data = json.loads('\n'.join([line for line in f]))

# Split the data into conversation data and expected values
conversations = [entry['conversation'] for entry in data]
expected_outputs = [entry['expected_output'] for entry in data]

print('SUCCESS: Number of conversations matches number of scores' if len(conversations) == len(expected_outputs) else 'FAIL: Number of conversations does not match number of scores')
print(f'Total conversations: {len(conversations)}')

In [ ]:
# Further split the data into training and test data
conv_train, _conv_test, score_train, _score_test = train_test_split(conversations, expected_outputs, random_state=1234)
print(f'Train: {len(conv_train)}, Test: {len(_conv_test)}')

In [ ]:
def Message(content, role='user'):
    return {
        'role': role,
        'content': content
    }

def create_prediction(transcript) -> str:
    completion = client.chat.completions.create(
        model=MODEL,
        temperature=0.1,  # low temperature for consistent scoring
        messages=[
            Message(SYSTEM_PROMPT, 'system'),
            Message(json.dumps(transcript))
        ],
    )
    return completion.choices[0].message.content

In [ ]:
# Run predictions on training and test set
def generate_scores(conversations):
    scores = []
    for i, c in enumerate(conversations):
        print(f'Generating {i+1}/{len(conversations)}')
        scores.append(create_prediction(c))
    return scores

train_scores_raw = generate_scores(conv_train)
test_scores_raw = generate_scores(_conv_test)

In [ ]:
# Parse json scores
def parse_scores(raw_scores):
    parsed = []
    for s in raw_scores:
        try:
            parsed.append(json.loads(s))
        except json.JSONDecodeError:
            print(f'Failed to parse: {s}')
            parsed.append({f'q{i}': '0' for i in range(1, 17)})
    return parsed

train_scores_parsed = parse_scores(train_scores_raw)
test_scores_parsed = parse_scores(test_scores_raw)

In [ ]:
from sklearn.metrics import mean_squared_error

def convert_scores_to_array(scores_df):
    return [int(s) if s else 0 for s in scores_df.values.flatten().tolist()]

def evaluate_split(parsed_scores, expected_scores, name):
    pred_df = pd.DataFrame(parsed_scores)
    expected_df = pd.DataFrame(expected_scores)

    # Ensure column order matches
    cols = [f'item{i}' for i in range(1, 17)]
    pred_df = pred_df[cols]
    expected_df = expected_df[cols]

    mse = mean_squared_error(
        convert_scores_to_array(expected_df),
        convert_scores_to_array(pred_df)
    )
    print(f'{name} MSE: {mse}')

    diff_df = expected_df.compare(pred_df, align_axis=1, keep_shape=True, keep_equal=True).rename(
        columns={'self': 'Expected', 'other': 'Predicted'}, level=1
    )
    display(diff_df)

evaluate_split(train_scores_parsed, score_train, 'Train')
evaluate_split(test_scores_parsed, _score_test, 'Test')

In [ ]:
# Derive domain scores from item scores
# D1 Sleep = max(item1, item2, item3, item4)
# D2 Sad = item5
# D3 Appetite/Weight = max(item6, item7, item8, item9)
# D4 Concentration = item10
# D5 Self-view = item11
# D6 Death/SI = item12
# D7 Interest = item13
# D8 Energy = item14
# D9 Psychomotor = max(item15, item16)

def compute_domain_scores(df):
    domains = pd.DataFrame()
    domains['D1_Sleep'] = df[['item1','item2','item3','item4']].astype(int).max(axis=1)
    domains['D2_Sad'] = df['item5'].astype(int)
    domains['D3_AppWt'] = df[['item6','item7','item8','item9']].astype(int).max(axis=1)
    domains['D4_Conc'] = df['item10'].astype(int)
    domains['D5_Self'] = df['item11'].astype(int)
    domains['D6_Death'] = df['item12'].astype(int)
    domains['D7_Interest'] = df['item13'].astype(int)
    domains['D8_Energy'] = df['item14'].astype(int)
    domains['D9_Psych'] = df[['item15','item16']].astype(int).max(axis=1)
    domains['Total'] = domains.sum(axis=1)
    return domains

def display_domain_scores(parsed_scores, expected_scores, name):
    print(f'\n--- {name} Domain Scores ---')
    pred_df = pd.DataFrame(parsed_scores)
    expected_df = pd.DataFrame(expected_scores)

    cols = [f'item{i}' for i in range(1, 17)]
    pred_df = pred_df[cols].fillna(0)
    expected_df = expected_df[cols].fillna(0)

    print(f'{name} Expected domain scores:')
    display(compute_domain_scores(expected_df))
    print(f'{name} Predicted domain scores:')
    display(compute_domain_scores(pred_df))

display_domain_scores(train_scores_parsed, score_train, 'Train')
display_domain_scores(test_scores_parsed, _score_test, 'Test')